# разбиение чисел 0...31 на 4 группы алгоритмом хебба

группы:
- 0-7 → `01`
- 8-15 → `10`
- 16-23 → `11`
- 24-31 → `00`

## кодируем число в 5 бит (-1/+1)

In [1]:
import numpy as np

def number_to_bits(n):
    result = []
    for i in [4, 3, 2, 1, 0]:
        bit = (n // (2 ** i)) % 2
        if bit == 0:
            result.append(-1)
        else:
            result.append(1)
    return result

for n in [0, 5, 15, 31]:
    print(f"число {n:2d} -> вход: {number_to_bits(n)}")

число  0 -> вход: [-1, -1, -1, -1, -1]
число  5 -> вход: [-1, -1, 1, -1, 1]
число 15 -> вход: [-1, 1, 1, 1, 1]
число 31 -> вход: [1, 1, 1, 1, 1]


## группы и коды

In [2]:
def get_group(n):
    return n // 8

group_codes = ["01", "10", "11", "00"]

for n in [0, 7, 8, 15, 16, 23, 24, 31]:
    g = get_group(n)
    print(f"число {n:2d} -> группа {g} (код {group_codes[g]})")

число  0 -> группа 0 (код 01)
число  7 -> группа 0 (код 01)
число  8 -> группа 1 (код 10)
число 15 -> группа 1 (код 10)
число 16 -> группа 2 (код 11)
число 23 -> группа 2 (код 11)
число 24 -> группа 3 (код 00)
число 31 -> группа 3 (код 00)


## обучающие пары

для каждого числа x - 5 бит, y - 4 числа (-1/+1), где +1 стоит на месте своей группы

In [3]:
training_data = []

for n in range(32):
    x = number_to_bits(n)
    g = get_group(n)
    y = [-1, -1, -1, -1]
    y[g] = 1
    training_data.append((x, y))

for n in [0, 3, 8, 20, 31]:
    x = number_to_bits(n)
    y = training_data[n][1]
    print(f"число {n:2d} ({group_codes[get_group(n)]}): x = {x}, y = {y}")

число  0 (01): x = [-1, -1, -1, -1, -1], y = [1, -1, -1, -1]
число  3 (01): x = [-1, -1, -1, 1, 1], y = [1, -1, -1, -1]
число  8 (10): x = [-1, 1, -1, -1, -1], y = [-1, 1, -1, -1]
число 20 (11): x = [1, -1, 1, -1, -1], y = [-1, -1, 1, -1]
число 31 (00): x = [1, 1, 1, 1, 1], y = [-1, -1, -1, 1]


## обучение по хеббу

W = сумма по всем примерам y * x^T, матрица 4x5

In [4]:
W = np.zeros((4, 5))

for x, y in training_data:
    x = np.array(x)
    y = np.array(y)
    W = W + np.outer(y, x)

print("матрица весов W:")
print(W)

матрица весов W:
[[-16. -16.   0.   0.   0.]
 [-16.  16.   0.   0.   0.]
 [ 16. -16.   0.   0.   0.]
 [ 16.  16.   0.   0.   0.]]


## проверка

считаем W * x и берём индекс максимального числа - это предсказанная группа

In [5]:
correct = 0

print(f"{'число':>6} | {'код':>4} | {'предсказано':>12} | {'верно':>6} | результат")
print("-" * 55)

for n in range(32):
    x = np.array(number_to_bits(n))
    output = W.dot(x)
    predicted_group = int(np.argmax(output))
    actual_group = get_group(n)

    if predicted_group == actual_group:
        correct = correct + 1
        status = "ok"
    else:
        status = "ошибка"

    print(f"{n:>6} | {group_codes[actual_group]:>4} | "
          f"{group_codes[predicted_group]:>12} | "
          f"{group_codes[actual_group]:>6} | {status}")

print("-" * 55)
print(f"правильно: {correct} из 32")

 число |  код |  предсказано |  верно | результат
-------------------------------------------------------
     0 |   01 |           01 |     01 | ok
     1 |   01 |           01 |     01 | ok
     2 |   01 |           01 |     01 | ok
     3 |   01 |           01 |     01 | ok
     4 |   01 |           01 |     01 | ok
     5 |   01 |           01 |     01 | ok
     6 |   01 |           01 |     01 | ok
     7 |   01 |           01 |     01 | ok
     8 |   10 |           10 |     10 | ok
     9 |   10 |           10 |     10 | ok
    10 |   10 |           10 |     10 | ok
    11 |   10 |           10 |     10 | ok
    12 |   10 |           10 |     10 | ok
    13 |   10 |           10 |     10 | ok
    14 |   10 |           10 |     10 | ok
    15 |   10 |           10 |     10 | ok
    16 |   11 |           11 |     11 | ok
    17 |   11 |           11 |     11 | ok
    18 |   11 |           11 |     11 | ok
    19 |   11 |           11 |     11 | ok
    20 |   11 |           11 |    

## итог

сеть выучила разбиение всех 32 чисел

по 4 выхода (а не 2) - потому что при 2 выходах данные симметричны и хебб даёт нулевые веса, а 4 выхода работают, коды 01/10/11/00 просто как имена групп

в матрице W последние два столбца нули - значит группа зависит только от двух старших бит, сеть это сама нашла